In [1]:
import fasttext


In [2]:
#train_file="merged_fasttext_data_cleaned.txt"

model=fasttext.train_supervised(
    input=train_file,
    epoch=10,
    lr=0.5,
    wordNgrams=2,
    verbose=2,
    dim=100,
    minCount=5,
    loss="ova"
)
model.save_model("issue_classifier_ft.bin")

# Evaluate accuracy
test_file="merged_fasttext_data_cleaned_test.txt"
result=model.test(test_file)
print("Number of examples:",result[0])
print("Precision:",result[1])
print("Recall:",result[2])
print("F1 score:",2*result[1]*result[2]/(result[1]+result[2]))

In [3]:
import fasttext


In [67]:
TARGETS = {
    "big": 500_000,      # for major classes
    "mid": (100_000, 200_000),  #----->><<<<< min–max range for medium classes
    "small": (50_000, 100_000)  # for rare issues
}

# Assign classes to tiers
BIG_CLASSES = {"__label__no_issue", "__label__refund_and_returns", "__label__product_quality", "__label__long_wait_times"}
MID_CLASSES = {"__label__delivery_delay", "__label__fake_counterfeit"}
SMALL_CLASSES = {"__label__packaging_handling", "__label__technical_support", "__label__order_issue", "__label__rare_issues", "__label__minor_issues", "__label__warranty_guarantee"}
import random
from collections import defaultdict

def parse_labels_and_text(line):
    parts = line.strip().split()
    labels = [p for p in parts if p.startswith("__label__")]
    text = " ".join([p for p in parts if not p.startswith("__label__")])
    return labels, text

def balance_dataset(input_file, output_file):
    # bucket rows
    buckets = defaultdict(list)
    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            labels, text = parse_labels_and_text(line)
            if not labels:
                continue
            key = labels[0]
            buckets[key].append(line.strip())

    balanced = []

    for label, rows in buckets.items():
        n = len(rows)

        if label in BIG_CLASSES:
            chosen = random.sample(rows, min(n, TARGETS["big"]))
            balanced.extend(chosen)

        elif label in MID_CLASSES:
            target = random.randint(*TARGETS["mid"])
            if n >= target:
                chosen = random.sample(rows, target)
                balanced.extend(chosen)
            else:
                # oversample if too small
                dup = random.choices(rows, k=target - n)
                balanced.extend(rows + dup)

        elif label in SMALL_CLASSES:
            target = random.randint(*TARGETS["small"])
            if n >= target:
                chosen = random.sample(rows, target)
                balanced.extend(chosen)
            else:
                dup = random.choices(rows, k=target - n)
                balanced.extend(rows + dup)

    random.shuffle(balanced)

    with open(output_file, "w", encoding="utf-8") as fout:
        fout.write("\n".join(balanced))

    print(f"✅ Balanced dataset written to {output_file}")
    print(f"Final size: {len(balanced):,} rows")
balance_dataset("train.txt", "train_balanced.txt")


✅ Balanced dataset written to train_balanced.txt
Final size: 2,778,108 rows


In [68]:
import fasttext

model = fasttext.train_supervised(
    input="train_balanced.txt",
    lr=0.5,
    epoch=25,
    wordNgrams=2,
    dim=200,
    loss="ova"   # one-vs-all, good for multi-label style imbalance
)

# Save your new strong model
model.save_model("issue_model.bin")


In [69]:
import fasttext

# load model --->>>>train
model = fasttext.load_model("issue_model.bin")

# ------------------------------------------------------------Validation evaluation
val_result = model.test("valid.txt")
print("Validation")
print("Samples:", val_result[0])
print("Precision@1:", val_result[1])
print("Recall@1:", val_result[2])

# Test evaluation
test_result = model.test("test.txt")
print("\nTest")
print("Samples:", test_result[0])
print("Precision@1:", test_result[1])
print("Recall@1:", test_result[2])
print("F1 score:::")



Validation
Samples: 706799
Precision@1: 0.9664699582200881
Recall@1: 0.8613298095012332

Test
Samples: 706620
Precision@1: 0.9668548866434575
Recall@1: 0.8613731811470956


In [71]:
f1 = 2 * test_result[1] * test_result[2] / (test_result[1] + test_result[2])
print(f"F1 Score: {f1:.4f}")

F1 Score: 0.9111
